In [3]:
# ===== Célula 1 — Carregar o modelo de embedding =====
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODELO = "intfloat/multilingual-e5-large-instruct"

inicio = time.perf_counter()
modelo = SentenceTransformer(MODELO, device="cpu")  # sua GPU é AMD: sem CUDA, usamos o processador
print("Precisão dos pesos ao carregar:", next(modelo.parameters()).dtype)

modelo = modelo.float()  # força float32: precisão total e o formato nativo do processador
print("Precisão usada:", next(modelo.parameters()).dtype)

print(f"Modelo carregado em {time.perf_counter() - inicio:.0f}s")
print(f"Cada texto vira um vetor de {modelo.get_embedding_dimension()} números")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Precisão dos pesos ao carregar: torch.float16
Precisão usada: torch.float32
Modelo carregado em 9s
Cada texto vira um vetor de 1024 números


In [5]:
# ===== Célula 2 — Ver como o modelo "entende" as frases =====
# Instrução da pergunta (em inglês, como no treinamento do modelo; a pergunta segue em português)
INSTRUCAO = ("Given a question about an online shopping experience, "
             "retrieve customer reviews that answer it")

def preparar_pergunta(texto: str) -> str:
    return f"Instruct: {INSTRUCAO}\nQuery: {texto}"

pergunta = "O que os clientes reclamam sobre atraso na entrega?"
avaliacoes = [
    "O produto chegou 20 dias depois do prazo",
    "Entrega demorou muito, fiquei semanas esperando",
    "Não recebi meu produto até hoje",
    "Chegou antes do prazo, recomendo",
    "Produto lindo, igual à foto",
    "O cartucho não é reconhecido pela impressora",
]

v_pergunta = modelo.encode([preparar_pergunta(pergunta)], normalize_embeddings=True)
v_avaliacoes = modelo.encode(avaliacoes, normalize_embeddings=True)

# Similaridade de cosseno: 1 = mesmo sentido; perto de 0 = sem relação
similaridade = (v_avaliacoes @ v_pergunta.T).ravel()
(pd.DataFrame({"avaliação": avaliacoes, "similaridade": similaridade.round(3)})
   .sort_values("similaridade", ascending=False))

,avaliação,similaridade
1,"Entrega demorou muito, fiquei semanas esperando",0.851
2,Não recebi meu produto até hoje,0.841
0,O produto chegou 20 dias depois do prazo,0.826
3,"Chegou antes do prazo, recomendo",0.820
4,"Produto lindo, igual à foto",0.782
5,O cartucho não é reconhecido pela impressora,0.755


In [6]:
# ===== Célula 3 — Medir a velocidade e estimar o tempo total =====
corpus = pd.read_parquet(RAIZ / "data" / "processed" / "corpus.parquet")
textos_distintos = corpus.drop_duplicates("chave_texto")
amostra = textos_distintos["texto"].sample(1000, random_state=42).tolist()

inicio = time.perf_counter()
vetores = modelo.encode(amostra, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
duracao = time.perf_counter() - inicio

velocidade = len(amostra) / duracao
total = len(textos_distintos)
print(f"\nVelocidade: {velocidade:.0f} textos por segundo")
print(f"Textos distintos a processar: {total:,}")
print(f"Estimativa para a base inteira: {total / velocidade / 60:.0f} minutos")
print(f"Formato do resultado da amostra: {vetores.shape}")

Batches:   0%|          | 0/16 [00:00<?, ?it/s]


Velocidade: 16 textos por segundo
Textos distintos a processar: 34,956
Estimativa para a base inteira: 36 minutos
Formato do resultado da amostra: (1000, 1024)


In [1]:
# ===== Célula 4 — Testar os índices: sentido com filtro, e palavra exata =====
import sys
from pathlib import Path

import bm25s
import chromadb
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))  # permite importar o pacote src a partir do notebook
from src.indexacao.embeddings import carregar_modelo, preparar_pergunta
from src.ingestao.construir_corpus import gerar_chave_texto

INDICE = RAIZ / "indice"
modelo = carregar_modelo()
colecao = chromadb.PersistentClient(path=str(INDICE / "chroma")).get_collection("avaliacoes")
print("Documentos no Chroma:", colecao.count())

# 1) Busca por SENTIDO com FILTRO de metadados
pergunta = "Quais problemas os clientes relatam sobre o produto?"
vetor = modelo.encode([preparar_pergunta(pergunta)], normalize_embeddings=True)
filtro = {"$and": [{"categoria_principal": "cama mesa banho"}, {"nota": {"$lte": 2}}]}
res = colecao.query(query_embeddings=vetor, n_results=5, where=filtro)
print(f"\nSentido + filtro (cama mesa banho, nota <= 2): '{pergunta}'")
for txt, meta, d in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
    print(f"  [sim {1 - d / 2:.3f}] nota {meta['nota']} | {meta['categoria_principal']} | {txt[:80]}")

# 2) Busca por PALAVRA EXATA (BM25)
bm25 = bm25s.BM25.load(str(INDICE / "bm25"))
ids_bm25 = pd.read_parquet(INDICE / "bm25_ids.parquet")["review_id"]
corpus = pd.read_parquet(RAIZ / "data" / "processed" / "corpus.parquet").set_index("review_id")

consulta = "correios"
tokens = bm25s.tokenize([gerar_chave_texto(consulta)], stopwords=None, show_progress=False)
posicoes, pontuacoes = bm25.retrieve(tokens, k=5, show_progress=False)
print(f"\nPalavra (BM25): '{consulta}'")
for pos, p in zip(posicoes[0], pontuacoes[0]):
    rid = ids_bm25.iloc[pos]
    print(f"  [bm25 {p:.2f}] nota {corpus.loc[rid, 'nota']} | {corpus.loc[rid, 'texto'][:80]}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Documentos no Chroma: 42251

Sentido + filtro (cama mesa banho, nota <= 2): 'Quais problemas os clientes relatam sobre o produto?'
  [sim 0.881] nota 1 | cama mesa banho | Produto entregue não condiz com o produto comprado, péssima qualidade, assessóri
  [sim 0.879] nota 1 | cama mesa banho | Péssima qualidade do produto
  [sim 0.878] nota 1 | cama mesa banho | ENGANANDO O CLIENTE COLOCA UMA FOTO RECEBE OUTRO PRODUTO , MATERIAL RUIM SE SOUB
  [sim 0.878] nota 1 | cama mesa banho | Falta de comprometimento. Comprei e as cores vieram todas erradas e escolhe com 
  [sim 0.877] nota 1 | cama mesa banho | Horrivel o atendimento. Fiz a compra, o produto não foi entregue,a nota foi emit

Palavra (BM25): 'correios'
  [bm25 2.75] nota 5 | Correios
  [bm25 2.40] nota 5 | Os correios atrasaram.
  [bm25 2.40] nota 4 | Correios não entregou
  [bm25 2.26] nota 1 | Esta nos correios desde 8/1
  [bm25 2.26] nota 5 | Problemas com os Correios
